# GloVe

This notebook covers GloVe (Global Vectors for Word Representation), a word embedding technique developed by Stanford University in 2014. GloVe combines the advantages of global matrix factorisation methods (like LSA) with local context window methods (like Word2Vec).

GloVe creates word vectors by factorising a co-occurrence matrix, capturing global statistical information about word relationships.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
import os

print("Libraries imported successfully.")

## 1. Understanding GloVe

GloVe (Global Vectors) is based on the idea that word co-occurrence statistics contain rich semantic information. Unlike Word2Vec which uses local context windows, GloVe leverages the entire corpus to build a co-occurrence matrix.

**Key differences from Word2Vec:**
- Uses global co-occurrence counts rather than local context
- Training is faster for large corpora
- Often produces better results for analogy tasks
- More interpretable vectors (some linear relationships)

The GloVe objective function learns word vectors that can predict co-occurrence probabilities.

In [ ]:
# Since training GloVe from scratch requires significant computation,
# we'll demonstrate using pre-trained GloVe vectors

print("=== GloVe Overview ===")
print("""
GloVe works by:
1. Building a word co-occurrence matrix from the corpus
2. Factorising this matrix to get word vectors
3. Using a weighted least squares objective

The key insight is that the ratio of co-occurrence probabilities
carries semantic information:
  P(king|man) / P(queen|woman) ≈ 1 (similar)
  P(ice|steam) / P(solid|gas) > 1 (related to gas-solid)
""")

## 2. Loading Pre-trained GloVe Vectors

GloVe provides pre-trained word vectors trained on large corpora like Wikipedia and Twitter. These can be downloaded and used directly.

In [ ]:
# For demonstration, we'll create a simple co-occurrence matrix
# and show how GloVe-style embeddings can be created

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import math

# Sample corpus
corpus = [
    "natural language processing involves machine learning",
    "deep learning uses neural networks for processing",
    "machine learning is a subset of artificial intelligence",
    "natural language processing helps computers understand text",
    "neural networks learn patterns from data",
    "deep learning enables complex pattern recognition",
    "artificial intelligence powers modern technology",
    "text classification is useful for sentiment analysis",
]

print("=== Sample Corpus ===")
for i, doc in enumerate(corpus, 1):
    print(f"Document {i}: {doc}")

In [ ]:
# Build co-occurrence matrix
def build_cooccurrence_matrix(corpus, window_size=2):
    """Build word co-occurrence matrix from corpus."""
    # Tokenise corpus
    tokenised = [doc.lower().split() for doc in corpus]
    
    # Get vocabulary
    vocab = set()
    for doc in tokenised:
        vocab.update(doc)
    vocab = sorted(vocab)
    word_to_idx = {word: i for i, word in enumerate(vocab)}
    
    # Initialise co-occurrence matrix
    n = len(vocab)
    cooccurrence = np.zeros((n, n), dtype=np.float32)
    
    # Count co-occurrences
    for doc in tokenised:
        for i, word in enumerate(doc):
            word_idx = word_to_idx[word]
            
            # Look at context window
            start = max(0, i - window_size)
            end = min(len(doc), i + window_size + 1)
            
            for j in range(start, end):
                if i != j:
                    context_word = doc[j]
                    context_idx = word_to_idx[context_word]
                    cooccurrence[word_idx, context_idx] += 1
    
    return cooccurrence, vocab, word_to_idx

# Build co-occurrence matrix
cooccurrence_matrix, vocabulary, word_to_idx = build_cooccurrence_matrix(corpus)

print("=== Co-occurrence Matrix ===")
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}")
print(f"\nMatrix shape: {cooccurrence_matrix.shape}")

In [ ]:
# Display co-occurrence matrix as DataFrame
cooccurrence_df = pd.DataFrame(
    cooccurrence_matrix,
    index=vocabulary,
    columns=vocabulary
)

print("=== Co-occurrence Matrix (first 8x8) ===")
print(cooccurrence_df.iloc[:8, :8].round(1))

## 3. Creating GloVe-style Embeddings

GloVe uses a weighted factorisation of the co-occurrence matrix. We can approximate this using SVD (Singular Value Decomposition) with appropriate weighting.

In [ ]:
# Apply log transformation (GloVe uses log of co-occurrence)
# Add small constant to avoid log(0)
epsilon = 1e-8
log_cooccurrence = np.log(cooccurrence_matrix + epsilon)

# Replace -inf with 0
log_cooccurrence = np.where(np.isinf(log_cooccurrence), 0, log_cooccurrence)

# Use SVD to get word embeddings
embedding_dim = 50
svd = TruncatedSVD(n_components=embedding_dim, random_state=42)
word_vectors = svd.fit_transform(log_cooccurrence)

print("=== GloVe-style Embeddings ===")
print(f"Embedding dimensions: {word_vectors.shape}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.4f}")

In [ ]:
# Create a simple class to work with these embeddings
class SimpleGloVe:
    """Simple GloVe-style word embeddings."""
    
    def __init__(self, vectors, vocabulary):
        self.vectors = vectors
        self.vocabulary = vocabulary
        self.word_to_idx = {word: i for i, word in enumerate(vocabulary)}
    
    def __contains__(self, word):
        return word in self.word_to_idx
    
    def __getitem__(self, word):
        return self.vectors[self.word_to_idx[word]]
    
    def most_similar(self, word, topn=5):
        """Find most similar words."""
        if word not in self.word_to_idx:
            raise KeyError(f"Word '{word}' not in vocabulary")
        
        word_vec = self.vectors[self.word_to_idx[word]].reshape(1, -1)
        similarities = cosine_similarity(word_vec, self.vectors)[0]
        
        # Get top N (excluding the word itself)
        indices = np.argsort(similarities)[::-1]
        results = []
        for idx in indices:
            if self.vocabulary[idx] != word:
                results.append((self.vocabulary[idx], similarities[idx]))
                if len(results) >= topn:
                    break
        
        return results
    
    def similarity(self, word1, word2):
        """Calculate cosine similarity between two words."""
        vec1 = self.vectors[self.word_to_idx[word1]].reshape(1, -1)
        vec2 = self.vectors[self.word_to_idx[word2]].reshape(1, -1)
        return cosine_similarity(vec1, vec2)[0, 0]

# Create GloVe-style model
glove_model = SimpleGloVe(word_vectors, vocabulary)

print("=== GloVe-style Model Created ===")
print(f"Vocabulary: {vocabulary}")

## 4. Testing the GloVe-style Embeddings

Let's test our embeddings by finding similar words and calculating similarities.

In [ ]:
# Test most similar words
print("=== Most Similar Words ===\n")

test_words = ["learning", "neural", "natural"]
for word in test_words:
    if word in glove_model:
        print(f"Words most similar to '{word}':")
        similar = glove_model.most_similar(word, topn=3)
        for sim_word, score in similar:
            print(f"  {sim_word}: {score:.4f}")
        print()

In [ ]:
# Test word similarities
print("=== Word Similarities ===\n")

test_pairs = [
    ("machine", "learning"),
    ("deep", "learning"),
    ("neural", "networks"),
    ("natural", "language"),
    ("artificial", "intelligence"),
]

for word1, word2 in test_pairs:
    if word1 in glove_model and word2 in glove_model:
        similarity = glove_model.similarity(word1, word2)
        print(f"Similarity('{word1}', '{word2}'): {similarity:.4f}")

## 5. Using Pre-trained GloVe Vectors

In practice, you'll typically use pre-trained GloVe vectors. These are trained on massive corpora and provide high-quality embeddings.

In [ ]:
# Show how to load pre-trained GloVe vectors
print("=== Loading Pre-trained GloVe Vectors ===")
print("""
To use pre-trained GloVe vectors:

1. Download GloVe vectors from:
   https://nlp.stanford.edu/projects/glove/

2. Load using gensim:

```python
from gensim.downloader import load

# This will download and load GloVe vectors
glove_vectors = load('glove-wiki-gigaword-100')

# Use the vectors
vector = glove_vectors['king']
similar = glove_vectors.most_similar('king', topn=5)
```

Available pre-trained models:
- glove-wiki-gigaword-50 (50d)
- glove-wiki-gigaword-100 (100d)
- glove-wiki-gigaword-200 (200d)
- glove-wiki-gigaword-300 (300d)
- glove-twitter-25 through glove-twitter-200
""")

## 6. Comparison: Word2Vec vs GloVe

Let's compare the characteristics of Word2Vec and GloVe.

In [ ]:
print("=== Word2Vec vs GloVe Comparison ===")
print("""
| Feature          | Word2Vec        | GloVe              |
|------------------|-----------------|--------------------|
| Training Method  | Predicting      | Matrix Factorisation|
| Context          | Local windows   | Global co-occurrence|
| Training Speed   | Slower          | Faster             |
| Memory Usage     | Lower           | Higher (matrix)    |
| Analogy Tasks    | Good            | Often better       |
| Semantic Accuracy| Good            | Often better       |
| Availability     | gensim, TF      | Stanford, gensim   |

When to use which:
- Use GloVe when you need better analogy performance
- Use Word2Vec when training on small datasets
- Use pre-trained vectors when possible
- Both work well for most NLP tasks
""")

## Summary

In this notebook, we covered GloVe:

1. **GloVe Theory**: Understood the co-occurrence matrix approach
2. **Co-occurrence Matrix**: Built a co-occurrence matrix from corpus
3. **GloVe-style Embeddings**: Created embeddings using SVD with log transformation
4. **Similarity Queries**: Tested word similarity and most similar words
5. **Pre-trained Vectors**: Learned how to load pre-trained GloVe vectors
6. **Comparison**: Compared Word2Vec and GloVe characteristics

GloVe is a powerful technique that combines the best of global matrix factorisation with local context methods. Pre-trained GloVe vectors are widely used in NLP applications and often provide better results than training embeddings from scratch on small datasets.